# MNIST Image dataset

The MNIST dataset contains digits from 0 to 9 written in different handstyles. This is a multiclass classifcation problem.
The code runs by itself only change is the path mydir (Documented below).

## installations
<ul>
<li>pip install torchvision (check last point on how to install the right way if you want to include cuda compatible)
<li>pip install torch
<li> pip install matplotlib (For plotting)
<li>go to https://pytorch.org/get-started/locally/?utm_source=chatgpt.com and download cuda version for your device.
</ul>




In [34]:
import kagglehub    
import numpy as np # linear algebra
from torchvision.models import resnet18, ResNet18_Weights
import torch

from torch import nn
import torchvision.datasets as datasets
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

from torchvision.transforms import ToTensor, Compose, RandomCrop

## DownLoad the dataset and conduct data agumentation and transformation to tensor. 
### Minst dataset consist of hand-written digits from 0 to 9 written in different hand-styles.
Download, convert to tensor, random crop the images

In [35]:
mydir=r"C:\Users\johan\Documents\Documents\School\advanced machine learning\labs\Computer-Vision\images\dataset" # this should be changed to a directory on your device
train_data=datasets.CIFAR10( 
    root=mydir,
    download=True,
    train=True,
    transform=Compose([
      
        ToTensor()
    ])
)
test_data=datasets.CIFAR10(
    root=mydir,
    download=True,
    train=False,
    transform=Compose([
       
        ToTensor()
    ])
)


### Check if cuda is available on device

In [36]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cpu


In [37]:

train_set,val_set=torch.utils.data.random_split(train_data, [5/6,1/6])


### Create a DataLoader with batch_size=64 

In [38]:
train_dataloader = DataLoader(train_set, batch_size=64, shuffle=True)
val_dataloader = DataLoader(val_set, batch_size=1000, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=64, shuffle=True)






# pre trained model resnet50


<li> Moves the module to cuda device so the module run the weights and parameters to GPU
<li> Chose loss-functions (CrossEntropyLoss because we have multiclass prediction)
<li> Chose optimizer and configure learning rate and momentum

In [ ]:
weights = ResNet18_Weights.DEFAULT
preprocess = weights.transforms()
model= resnet18(weights=ResNet18_Weights).to(device)
for param in model.parameters():
    param.requires_grad=False

model.fc = nn.Linear(model.fc.in_features, 10)

loss_fn=nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=0.001)




Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\johan/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


C:\Users\johan\AppData\Roaming\Python\Python312\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100%|██████████| 44.7M/44.7M [00:03<00:00, 11.8MB/s]


## Training function
<li> Moves input and output to cuda device.
<li> Set the gradient to Nan after every batch to not include past behaviour (accumulated calcultaed gradient )in new training patch. 
<li> Predict x
<li> Calculate loss
<li> loss.backward() computes the gradient of current tensor. (calculate the gradient for every parameter w.r.t loss) 
<li> Optimizer.step() Update parameters based on the calculated gradient


In [40]:
def training_loop(module , train_loader, preprocess, val_dataloader, loss_fn, optimizer=None ):
    
    if optimizer == None:
        optimizer = torch.optim.SGD(module.parameters(), lr=0.001, momentum=0.9)
    module.train()
    running_loss = 0.0
    total_loss=0.0
    for batch, (X,y) in enumerate(train_loader):
        X=preprocess(X)
        X, y= X.to(device), y.to(device)
        optimizer.zero_grad()
        pred=module(X)
        loss=loss_fn(pred,y)
        loss.backward()
        optimizer.step()
        running_loss +=loss.item()
        total_loss +=loss.item()
        if batch % 100 == 0:    
            print(f'[ {batch + 1:5d}] loss: {running_loss / 100:.3f}')
            running_loss = 0.0


    module.eval()
    with torch.no_grad():
        total_val_loss=0
        for im, l in val_dataloader:
            X_val=im.to(device) 
            y_val=l.to(device)
       
            val_pred=module(X_val)
            val_loss=loss_fn(val_pred,y_val)
            
            total_val_loss+=val_loss
    
        print(f'training_loss: {total_loss:.3f}: validaton_loss: {total_val_loss:.3f}')


    print('Finished Training')
       

## Test function
Make prediction on unseen data then calcuulate accuracy

In [41]:
def test_loop(module, dataloader, loss_fn):
    module.eval()
    correct = 0
    total = 0
    total_loss = 0.0

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = module(images)
            loss = loss_fn(outputs, labels)

            total_loss += loss.item()

            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total

    print(f'Accuracy of the network on the test images: {accuracy:.2f}%, Test loss: {avg_loss:.3f}')

### Run the module
<li> Define number of epoch and run training


In [44]:
epoch=10
for i in range(epoch):
    training_loop(model,train_dataloader,preprocess,val_dataloader,loss_fn,optimizer)

[     1] loss: 0.011
[   101] loss: 0.727
[   201] loss: 0.726
[   301] loss: 0.729
[   401] loss: 0.706
[   501] loss: 0.687
[   601] loss: 0.684
training_loss: 463.092: validaton_loss: 32.599
Finished Training
[     1] loss: 0.007
[   101] loss: 0.687
[   201] loss: 0.682
[   301] loss: 0.683
[   401] loss: 0.674
[   501] loss: 0.645
[   601] loss: 0.693
training_loss: 441.888: validaton_loss: 34.939
Finished Training
[     1] loss: 0.006
[   101] loss: 0.681
[   201] loss: 0.660
[   301] loss: 0.663
[   401] loss: 0.647
[   501] loss: 0.653
[   601] loss: 0.656
training_loss: 431.699: validaton_loss: 37.395
Finished Training
[     1] loss: 0.007
[   101] loss: 0.652
[   201] loss: 0.636
[   301] loss: 0.636
[   401] loss: 0.640
[   501] loss: 0.652
[   601] loss: 0.660
training_loss: 423.775: validaton_loss: 40.250
Finished Training
[     1] loss: 0.007
[   101] loss: 0.655
[   201] loss: 0.648
[   301] loss: 0.651
[   401] loss: 0.641
[   501] loss: 0.605
[   601] loss: 0.654
train

## Run the module on test data


In [45]:
test_loop(model, test_dataloader,loss_fn)

Accuracy of the network on the test images: 13.20%, Test loss: 4.944
